In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [0]:
%pip install kaggle

In [0]:
import os

os.environ['KAGGLE_USERNAME'] = 'radityaja' # Pakai username Kaggle sendiri
os.environ['KAGGLE_KEY'] = 'dddcfd86360cf5b12672873749e6ad92'

In [0]:
import subprocess
import os

download_path = "/Volumes/workspace/default/nyc_taxi_volume"
os.chdir(download_path)

# Download train.csv file
subprocess.run([
    "kaggle", "competitions", "download",
    "-c", "new-york-city-taxi-fare-prediction",
    "-f", "train.csv"
], check=True)

print(f"Download selesai! File tersimpan di {download_path}")

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

In [0]:
import zipfile

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

zip_path   = f"{VOLUME_PATH}/train.csv.zip"
extract_to = VOLUME_PATH

print("Mulai unzip...")
with zipfile.ZipFile(zip_path, 'r') as z:
    members = z.namelist()
    print(f"File di dalam zip: {members}")
    z.extractall(extract_to)
    print(f"Extracted ke: {extract_to}")

print("\nIsi Volume setelah unzip:")
for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

In [0]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Full_Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

schema = StructType([
    StructField("key",               StringType(),  True),
    StructField("fare_amount",       DoubleType(),  True),
    StructField("pickup_datetime",   StringType(),  True),
    StructField("pickup_longitude",  DoubleType(),  True),
    StructField("pickup_latitude",   DoubleType(),  True),
    StructField("dropoff_longitude", DoubleType(),  True),
    StructField("dropoff_latitude",  DoubleType(),  True),
    StructField("passenger_count",   IntegerType(), True),
])

def checkpoint(df, name):
    path = f"{VOLUME_PATH}/checkpoints/{name}"
    df.write.mode("overwrite").parquet(path)
    df_loaded = spark.read.parquet(path)
    print(f"Checkpoint '{name}' disimpan & dimuat ulang.")
    return df_loaded

print("Loading train.csv...")
df_raw = spark.read.csv(
    f"{VOLUME_PATH}/train.csv",
    header=True,
    schema=schema
)
df_raw.show(5)

In [0]:
print("Jumlah data:", df_raw.count())
print("Jumlah kolom:", len(df_raw.columns))
df_raw.printSchema()
df_raw.explain()

In [0]:
df_raw.describe().show()

In [0]:
df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
]).show()

In [0]:
df_clean = df_raw.dropna(subset=[
    "dropoff_longitude",
    "dropoff_latitude"
])

In [0]:
total_rows = df_raw.count()
unique_keys = df_raw.select("key").distinct().count()

print("Total:", total_rows)
print("Duplikat:", total_rows - unique_keys)

In [0]:
total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates().count()

print("Total:", total_rows)
print("Duplikat:", total_rows - unique_rows)

In [0]:
df_dup_keys = df_raw.groupBy("key") \
                   .count() \
                   .filter("count > 1") \
                   .select("key")

df_full_dup = df_raw.join(df_dup_keys, on="key", how="inner")

df_full_dup.show(truncate=False)

In [0]:
df_base = (
    df_raw
    .drop("key")
    .dropna()
)

df_base.show(5)

In [0]:
df_time = (
    df_base
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime", "yyyy-MM-dd HH:mm:ss 'UTC'"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("pickup_dayofweek", F.dayofweek("pickup_ts"))
    .withColumn("pickup_month", F.month("pickup_ts"))
    .withColumn("pickup_year", F.year("pickup_ts"))
)

df_time.show(5)

In [0]:
R = 6371

df_distance = (
    df_time
    .withColumn(
        "distance_km",
        2 * R * atan2(
            sqrt(
                sin((radians(col("dropoff_latitude") - col("pickup_latitude")) / 2))**2 +
                cos(radians(col("pickup_latitude"))) *
                cos(radians(col("dropoff_latitude"))) *
                sin((radians(col("dropoff_longitude") - col("pickup_longitude")) / 2))**2
            ),
            sqrt(
                1 - (
                    sin((radians(col("dropoff_latitude") - col("pickup_latitude")) / 2))**2 +
                    cos(radians(col("pickup_latitude"))) *
                    cos(radians(col("dropoff_latitude"))) *
                    sin((radians(col("dropoff_longitude") - col("pickup_longitude")) / 2))**2
                )
            )
        )
    )
)

df_distance.select("distance_km").show(5)

In [0]:
df_clean = (
    df_distance
    .filter((col("distance_km") > 0) & (col("distance_km") < 100))
    .filter((col("fare_amount") > 0) & (col("fare_amount") < 500))
    .filter((col("passenger_count").between(1, 6)))
    .filter(col("pickup_longitude").between(-74.2591, -73.7004))
    .filter(col("pickup_latitude").between(40.4774, 40.9176))
    .filter(col("dropoff_longitude").between(-74.2591, -73.7004))
    .filter(col("dropoff_latitude").between(40.4774, 40.9176))
)

df_clean.show(5)

In [0]:
df_final = (
    df_clean
    .withColumn(
        "day_type",
        F.when(F.col("pickup_dayofweek").isin(1,7), "weekend")
         .otherwise("weekday")
    )
    .withColumn(
        "rush_hour",
        F.when((F.col("pickup_hour").between(7,9)) | (F.col("pickup_hour").between(16,19)), 1)
         .otherwise(0)
    )
    .withColumn(
        "time_of_day",
        F.when(col("pickup_hour") < 6, "night")
         .when(col("pickup_hour") < 12, "morning")
         .when(col("pickup_hour") < 18, "afternoon")
         .otherwise("evening")
    )
    .withColumn(
        "rush_hour",
        F.when(
            (col("pickup_hour").between(7,9)) | 
            (col("pickup_hour").between(16,19)), 1
        ).otherwise(0)
    )
    .withColumn(
        "trip_type",
        F.when(col("distance_km") < 2, "short")
         .when(col("distance_km") < 10, "medium")
         .otherwise("long")
    )
)

df_final.show(5)

In [0]:
df_final.select(
    sum((col("fare_amount") <= 0).cast("int")).alias("fare_anomali"),

    sum(((col("passenger_count") <= 0) | (col("passenger_count") > 6)).cast("int"))
        .alias("passenger_anomali"),

    sum(((col("distance_km") == 0) | (col("distance_km") > 100)).cast("int"))
        .alias("distance_anomali"),

    sum(((col("pickup_longitude") < -74.2591) | (col("pickup_longitude") > -73.7004)).cast("int"))
        .alias("pickup_longitude_anomali"),

    sum(((col("dropoff_longitude") < -74.2591) | (col("dropoff_longitude") > -73.7004)).cast("int"))
        .alias("dropoff_longitude_anomali"),

    sum(((col("pickup_latitude") < 40.4774) | (col("pickup_latitude") > 40.9176)).cast("int"))
        .alias("pickup_latitude_anomali"),

    sum(((col("dropoff_latitude") < 40.4774) | (col("dropoff_latitude") > 40.9176)).cast("int"))
        .alias("dropoff_latitude_anomali")

).show()

In [0]:
df_final.groupBy("day_type") \
   .agg(
       F.min("fare_amount").alias("min"),
       F.max("fare_amount").alias("max"),
       F.avg("fare_amount").alias("avg")
   ) \
   .show()

In [0]:
df_clean.groupBy("passenger_count") \
        .count() \
        .orderBy(col("count").desc()) \
        .show()

In [0]:
df_clean.select("fare_amount", "distance_km").show(10)

In [0]:
cols = [
    "fare_amount",
    "distance_km",
    "passenger_count",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "pickup_year"
]

In [0]:
df_sample = df_final.select(cols).sample(0.1).toPandas()

In [0]:
corr = df_sample.corr()

In [0]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    linecolor="white"
)

plt.title("Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_sample,
    x="distance_km",
    y="fare_amount",
    alpha=0.3
)
plt.title("Fare vs Distance")
plt.show()